# Experiments for the survey paper

This notebook produces the real measurements for the empirical section of the survey
*Representation-Aware Vector Search*. It embeds the **Food-101** images with **six public backbones**, one per
training paradigm (supervised CNN, supervised CNN pretrained on ImageNet-22k, supervised ViT, CLIP, DINOv2, MAE), and
measures:

| Test | What is measured |
|---|---|
| **Geometry cards** | One card per backbone: intrinsic dimension, contrast, hubness, anisotropy, clusters, query shift, compression profile (PCA, 8-bit, 4-bit, 1-bit), reference HNSW cost |
| **P1** | At the same dimension (PCA to 256), do query difficulty measures rank the backbones by search work? |
| **P2** | With one CLIP model, how much more work do text queries need than image queries on the same index, and is it the shift or the difficulty of the queries? |
| **P3** | How search work at a fixed end-to-end recall changes as the dimension falls (native, 256, 128, 64), and where its minimum is |
| **P4, P5** | At 256 dimensions, what a hubness-reducing head and a KoLeo-spreading head change compared with a plain neighbour-ranking head |

**You do not need to download anything.** Food-101 is public and is downloaded automatically from Hugging Face
(huggingface.co/datasets/ethz/food101, about 5 GB). The pretrained weights are downloaded automatically too.

**Before you start:** *Runtime → Change runtime type → T4 GPU*, then *Runtime → Run all*.

**Time:** several hours (not yet measured on Colab). If Colab disconnects, reconnect and *Run all* again: everything is
copied to `MyDrive/RAVI_survey` every 10 minutes and after each stage, and finished work is skipped.

**At the end** the file `survey_results.zip` (a few MB) is downloaded. Send that file back for the write-up; it holds
all the tables and figures. Do not edit any number in it.

In [ ]:
# ---- settings -------------------------------------------------------------
USE_DRIVE = True          # keep code and results on Google Drive so a disconnect loses nothing
DRIVE_DIR = "/content/drive/MyDrive/RAVI_survey"
RUN_CROSSMODAL = True     # P2 (text vs image queries with CLIP); adds about 15-30 minutes
OVERWRITE_CODE = False    # True replaces the code saved on Drive with the version embedded in this notebook
RUN_UNIT_TESTS = True     # about one minute; checks metrics and indexes on cases with known answers
SYNC_MINUTES = 10         # how often a running stage copies its new files to Drive

import os
TEST_MODE = os.environ.get("RAVI_NOTEBOOK_TEST") == "1"   # offline self-test only; leave unset in Colab
if TEST_MODE:   # a local folder stands in for Google Drive
    DRIVE_DIR, SYNC_MINUTES = os.path.abspath("fake_drive"), 0.5

## 1. Environment

In [ ]:
import os, sys, shutil, subprocess
IN_COLAB = "google.colab" in sys.modules
print("Python", sys.version.split()[0], "| CPU cores:", os.cpu_count())
try:
    import torch
    gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
    print("torch", torch.__version__, "| GPU:", gpu or "none")
except ImportError:
    gpu = None
    print("torch is not installed")
free = shutil.disk_usage("/content" if os.path.exists("/content") else "/").free
print(f"free local disk: {free / 1e9:.0f} GB")
if not TEST_MODE and gpu is None:
    print("\nWARNING: no GPU. Use Runtime > Change runtime type > T4 GPU, then run all cells again.")

In [ ]:
if USE_DRIVE and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
BASE = DRIVE_DIR if USE_DRIVE else os.path.abspath("ravi_work")          # code, copies of the run, result archive
LOCAL_RUNS = "/content/ravi_runs" if IN_COLAB else os.path.abspath("ravi_local_runs")   # where stages write
os.makedirs(BASE, exist_ok=True)
print("saved folder:", BASE, "| working folder (fast local disk):", LOCAL_RUNS)

## 2. Install and unpack the code

In [ ]:
if not TEST_MODE:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "faiss-cpu>=1.8.0", "timm>=1.0.9",
                    "datasets>=2.19", "pyarrow", "pyyaml", "psutil", "pytest", "open_clip_torch>=2.24"], check=True)
print("packages ready")

In [ ]:
# @title Code archive (the RAVI package, configs and tests; do not edit)
CODE_ZIP_B64 = "..."

In [ ]:
import base64, io, zipfile
CODE_DIR = os.path.join(BASE, "ravi_code")
if OVERWRITE_CODE or not os.path.exists(os.path.join(CODE_DIR, "ravi", "survey.py")):
    zipfile.ZipFile(io.BytesIO(base64.b64decode(CODE_ZIP_B64))).extractall(CODE_DIR)
    print("code written to", CODE_DIR)
else:
    print("using the code already in", CODE_DIR, "(set OVERWRITE_CODE = True to replace it)")
sys.path.insert(0, CODE_DIR)
import ravi
print("ravi version", ravi.__version__)
if RUN_UNIT_TESTS:
    r = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/test_core.py", "tests/test_sync.py",
                        "tests/test_survey.py"], cwd=CODE_DIR, capture_output=True, text=True)
    print(r.stdout[-600:] or r.stderr[-600:])
    if r.returncode != 0:
        raise RuntimeError("unit tests failed; see the output above")

## 3. Configure the run

In [ ]:
if TEST_MODE:   # offline self-test: a small fake image dataset (20 classes of coloured patterns)
    import numpy as np, datasets
    from PIL import Image
    rng = np.random.default_rng(0)
    imgs, labels = [], []
    for c in range(20):
        base_col = rng.integers(0, 255, 3)
        for _ in range(132):
            a = np.clip(base_col + rng.normal(0, 40, (64, 64, 3)), 0, 255).astype(np.uint8)
            imgs.append(Image.fromarray(a)); labels.append(c)
    feats = datasets.Features({"image": datasets.Image(), "label": datasets.ClassLabel(num_classes=20)})
    os.makedirs("/tmp/ravi_fake", exist_ok=True)
    datasets.Dataset.from_dict({"image": imgs, "label": labels}, features=feats).to_parquet("/tmp/ravi_fake/fake.parquet")

import yaml, contextlib
from ravi.sync import PeriodicPush, restore
cfg = yaml.safe_load(open(os.path.join(CODE_DIR, "configs", "colab_survey_test.yaml" if TEST_MODE else "colab_survey.yaml")))
RUN_DIR = os.path.join(LOCAL_RUNS, cfg["name"])                       # every stage writes here
DRIVE_RUN = os.path.join(BASE, "runs", cfg["name"]) if USE_DRIVE else None
cfg["out_dir"] = RUN_DIR
if TEST_MODE:
    for split in ("base", "query"):
        cfg["dataset"][split]["data_files"] = "/tmp/ravi_fake/fake.parquet"
        cfg["dataset"][split]["cache_dir"] = "/tmp/ravi_fake/cache"
if DRIVE_RUN and os.path.isdir(DRIVE_RUN):
    print("copying this run back from Drive (only files missing on this machine) ...")
    r = restore(DRIVE_RUN, RUN_DIR)
    print(f"  {r['files']} files, {r['bytes'] / 1e9:.2f} GB restored")
os.makedirs(RUN_DIR, exist_ok=True)
CONFIG = os.path.join(RUN_DIR, "config.yaml")
if not os.path.exists(CONFIG):
    yaml.safe_dump(cfg, open(CONFIG, "w"), sort_keys=False)
else:
    print("keeping the existing config of this run (delete runs/" + cfg["name"] + " on Drive to start afresh)")
print("working folder:", RUN_DIR, "| copied to:", DRIVE_RUN or "(Drive not used)")
print("backbones:", [m["name"] for m in yaml.safe_load(open(CONFIG))["models"]])

def ravi_stage(stage):
    """Run one pipeline stage, stream its log and copy new files to Drive while it runs and when it ends.
    Safe to re-run: finished work is skipped."""
    env = dict(os.environ, PYTHONPATH=CODE_DIR, PYTHONUNBUFFERED="1")
    sync = PeriodicPush(RUN_DIR, DRIVE_RUN, every_s=SYNC_MINUTES * 60) if DRIVE_RUN else contextlib.nullcontext()
    with sync:
        p = subprocess.Popen([sys.executable, "-m", "ravi", stage, "-c", CONFIG], cwd=CODE_DIR, env=env,
                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        try:
            for line in p.stdout:
                print(line, end="")
            code = p.wait()
        except BaseException:          # cell interrupted: stop the stage before the last copy to Drive
            p.terminate()
            p.wait()
            raise
    if code != 0:
        raise RuntimeError(f"stage '{stage}' failed (see the log above). Fix the cause and run the cell again; "
                           "finished work is kept.")

ravi_stage("plan")

## 4. Run the experiments

Each cell is one stage. After a disconnect, run all cells again from the top: finished work is skipped.

**4.1 Embeddings.** Downloads Food-101 and the six sets of pretrained weights, then embeds every image on the GPU.

In [ ]:
ravi_stage("extract")

**4.2 P2: text and image queries with CLIP.** Embeds the images and 808 text prompts (101 dishes x 8 wordings) with one CLIP model, then searches one HNSW index with both kinds of query.

In [ ]:
if RUN_CROSSMODAL:
    ravi_stage("crossmodal")
else:
    print("skipped (RUN_CROSSMODAL = False)")

**4.3 Compact spaces.** PCA to 64, 128 and 256 dimensions for every backbone, and the three heads (plain ranking, hubness-reducing, KoLeo-spreading) at 256 dimensions.

In [ ]:
ravi_stage("reduce")

**4.4 Exact neighbours** of every query in every space, and how many native neighbours each compact space keeps.

In [ ]:
ravi_stage("gt")

**4.5 Geometry** of every space: intrinsic dimension, contrast, hubness, anisotropy, clusters, query shift.

In [ ]:
ravi_stage("geometry")

**4.6 Index benchmark** (the longest stage): HNSW and IVF in every space, two seeds. Every build is saved as soon as it finishes.

In [ ]:
ravi_stage("bench")

**4.7 Analysis:** tables, statistics, figures, and the survey measurements (`results/survey_report.md`).

In [ ]:
ravi_stage("analyse")

## 5. Results

Everything below is read from this run's files. With six backbones, correlations are an illustration: only
|rho| of about 0.89 or more reaches p < 0.05.

In [ ]:
from IPython.display import Markdown, Image, display
res = os.path.join(RUN_DIR, "results")
display(Markdown(open(os.path.join(res, "survey_report.md")).read()))

In [ ]:
fig_dir = os.path.join(RUN_DIR, "figures")
for name in ("survey_p1", "survey_p2", "survey_p3"):
    f = os.path.join(fig_dir, name + ".png")
    if os.path.exists(f):
        print(name)
        display(Image(f, width=620))

## 6. Save and download the results
`survey_results.zip` holds the configuration, all result tables (`results/`), the figures, the raw measurement of every
index build (`parts/`) and the image lists. The embeddings stay on Drive and are not included.

In [ ]:
import zipfile
archive = os.path.join(LOCAL_RUNS, "survey_results.zip")
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(CONFIG, "config.yaml")
    for sub in ("results", "figures", "parts", "lists"):
        for d, _, fs in os.walk(os.path.join(RUN_DIR, sub)):
            for f in fs:
                if ".tmp" not in f and ".partial" not in f:
                    z.write(os.path.join(d, f), os.path.relpath(os.path.join(d, f), RUN_DIR))
    meta = os.path.join(RUN_DIR, "crossmodal", "meta.json")
    if os.path.exists(meta):
        z.write(meta, "crossmodal/meta.json")
if USE_DRIVE:
    shutil.copy(archive, os.path.join(BASE, os.path.basename(archive)))
    print("saved to Drive:", os.path.join(BASE, os.path.basename(archive)))
print(f"{os.path.basename(archive)}: {os.path.getsize(archive) / 1e6:.1f} MB")
if IN_COLAB:
    from google.colab import files
    files.download(archive)

## What to do with the results

Send `survey_results.zip` back. The survey's empirical section will be written only from the numbers in it, with the
dataset, backbones, hardware and number of seeds stated, and with the limits of six backbones on one dataset made
explicit. Search work is reported as distance evaluations per query, which does not depend on Colab's shared hardware;
Colab timings are not used for claims.

If a prediction is **not** supported by the measurements, that is a result too and will be reported as such.